# Lezione 08: Tipi e fonti di dati

**Obiettivi di apprendimento:**
- Classificare un dato come strutturato, semi-strutturato o non strutturato.
- Individuare fonti di dati comuni (database, API, web scraping, stream).
- Leggere CSV/JSON con pandas.

**Durata stimata:** 110 min

**Capitoli del manuale:**
- [09 — Tipi di dati](../manual/09_tipos_de_datos_IT.md)
- [10 — Da dove vengono i dati](../manual/10_fuentes_de_datos_apis_IT.md)


## Parte 1: Tipi di dati

Prima di analizzare qualsiasi dato, devi sapere con che tipo di dato hai a che fare. Ci sono tre categorie:

- **Strutturati**: schema fisso, formato tabulare (SQL, fogli di calcolo, sensori GPS).
- **Semi-strutturati**: organizzazione parziale tramite etichette/metadati, senza schema fisso (XML, JSON, email).
- **Non strutturati**: senza schema identificabile (testo libero, immagini, audio, video, PDF).

Costruiamo un esempio di ognuno e vediamo come cambia il modo di lavorare con ogni tipo.

In [ ]:
import pandas as pd
import numpy as np
import json
import io
import sqlite3
from collections import Counter

# --- 1. Dati STRUTTURATI ---
ventas_estructuradas = pd.DataFrame({
    "id_pedido": [1, 2, 3, 4],
    "cliente": ["Ana", "Luis", "Marta", "Ana"],
    "importe": [59.90, 120.00, 15.50, 42.30],
})
print("--- Strutturati ---")
print(ventas_estructuradas)
print(ventas_estructuradas.dtypes)


In [ ]:
# --- 2. Dati SEMI-STRUTTURATI ---
# Simuliamo la risposta tipica di un'API: JSON con struttura annidata
pedidos_json = '''
[
  {"id_pedido": 1, "cliente": "Ana", "productos": ["libro", "taza"]},
  {"id_pedido": 2, "cliente": "Luis", "productos": ["portatil"]},
  {"id_pedido": 3, "cliente": "Marta", "productos": ["libro", "cuaderno", "boli"]}
]
'''

pedidos_dict = json.loads(pedidos_json)
print("--- Semi-strutturati (JSON grezzo) ---")
print(pedidos_dict)

# json_normalize appiattisce la struttura in un DataFrame
pedidos_df = pd.json_normalize(pedidos_dict)
pedidos_df["n_productos"] = pedidos_df["productos"].apply(len)
print("\n--- Semi-strutturati (come DataFrame) ---")
print(pedidos_df)


In [ ]:
# --- 3. Dati NON STRUTTURATI ---
comentarios = [
    "Muy buen servicio, llego rapido",
    "El producto llego danado, pesima experiencia",
    "Precio correcto, volveria a comprar",
    "Rapido y muy buen precio, encantada",
]

print("--- Non strutturati (testo libero) ---")
for c in comentarios:
    print("-", c)

# Prima di analizzarlo con pandas bisogna decidere COSA misureremo:
# qui, contiamo le parole per commento e la frequenza globale
longitudes = [len(c.split()) for c in comentarios]
print("\nParole per commento:", longitudes)

todas_las_palabras = " ".join(comentarios).lower().split()
frecuencia = Counter(todas_las_palabras)
print("Parole più comuni:", frecuencia.most_common(5))


Nota lo sforzo crescente: i dati strutturati erano già pronti; quelli semi-strutturati hanno richiesto un passo di conversione (`json.loads` + `json_normalize`); quelli non strutturati hanno richiesto che decidessimo noi stessi cosa misurare (qui, contare le parole) prima che assomigliassero in qualche modo a una tabella.

## Parte 2: Da dove vengono i dati?

Ora l'altra metà della domanda: da dove escono quei dati? Fonti tipiche:

- **Database relazionali interni**: SQL Server, Oracle, MySQL, IBM DB2.
- **File piatti e XML**: CSV (il più comune), fogli di calcolo, XML (gerarchico).
- **API e servizi web**: social media, mercati borsistici, validazione di dati — di solito restituiscono JSON.
- **Web scraping**: BeautifulSoup, Scrapy, Pandas, Selenium — quando non c'è un'API disponibile.
- **Stream di dati**: Apache Kafka, Spark Streaming, Apache Storm — dati che arrivano in modo continuo (sensori IoT, transazioni, ticker di borsa).

Simuliamo tre fonti diverse —CSV, JSON di un'API, e un database relazionale— e le combiniamo, che è ciò che fa un data scientist nella pratica quasi sempre.

In [ ]:
# --- Fonte 1: file piatto (CSV) ---
csv_texto = '''id_cliente,nombre,ciudad,gasto_total
1,Ana,Madrid,340.50
2,Luis,Sevilla,120.00
3,Marta,Bilbao,89.90
'''
clientes_csv = pd.read_csv(io.StringIO(csv_texto))
print("--- Fonte: CSV ---")
print(clientes_csv)


In [ ]:
# --- Fonte 2: API (JSON) ---
json_texto = '''
[
  {"cp": "28001", "ciudad": "Madrid", "provincia": "Madrid"},
  {"cp": "41001", "ciudad": "Sevilla", "provincia": "Sevilla"},
  {"cp": "48001", "ciudad": "Bilbao", "provincia": "Bizkaia"}
]
'''
codigos_postales = pd.read_json(io.StringIO(json_texto))
print("--- Fonte: API (JSON) ---")
print(codigos_postales)


In [ ]:
# --- Fonte 3: database relazionale (sqlite3 in memoria) ---
conexion = sqlite3.connect(":memory:")
cursor = conexion.cursor()

cursor.execute('''
    CREATE TABLE pedidos (
        id_pedido INTEGER,
        id_cliente INTEGER,
        importe REAL
    )
''')
cursor.executemany(
    "INSERT INTO pedidos VALUES (?, ?, ?)",
    [(1, 1, 59.90), (2, 2, 120.00), (3, 1, 15.50), (4, 3, 42.30)],
)
conexion.commit()

pedidos_db = pd.read_sql_query("SELECT * FROM pedidos", conexion)
print("--- Fonte: database relazionale ---")
print(pedidos_db)

conexion.close()


In [ ]:
# --- Combinando le tre fonti ---
combinado = pedidos_db.merge(clientes_csv, on="id_cliente")
resumen = combinado.groupby("ciudad")["importe"].sum()
print("--- Spesa totale per città, combinando DB + CSV ---")
print(resumen)


## Riepilogo della lezione

- **Strutturati** (schema fisso, tabelle), **semi-strutturati** (JSON/XML, gerarchici), **non strutturati** (testo libero, immagini): ognuno richiede uno strumento diverso.
- I dati arrivano da database relazionali, file piatti, API, web scraping o stream in tempo reale.
- Non importa l'origine: il lavoro del data scientist è sempre convertire i dati in un formato comune (di solito tabulare) per poterli incrociare e analizzare, come abbiamo fatto combinando il database con il CSV.
- `pandas.read_csv`, `pandas.read_json` e `pandas.read_sql_query` sono le tre porte d'ingresso più abituali a un DataFrame.

**Materiale correlato:**
- Capitoli del manuale: [09 — Tipi di dati](../manual/09_tipos_de_datos_IT.md) e [10 — Da dove vengono i dati](../manual/10_fuentes_de_datos_apis_IT.md)
- Notebook di pratica: [`clase08_practica_tipos_y_fuentes_de_datos_IT.ipynb`](../notebooks_practica/clase08_practica_tipos_y_fuentes_de_datos_IT.ipynb)
